In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.stattools import acf

OUTPUT_DIR  = Path("../outputs")

# Diagnostics

This notebook loads the raw outputs produced by main.py and evaluates the quality of the Gibbs sampler chain.
It does NOT run the model — run main.py first, then open this.

What it is checked:
   - whether the chain has mixed and converged (trace plots, ESS)
   - whether individual edges are well-identified (autocorrelation)
   - the posterior distribution of each Gibbs check (inclusion probs, MAP graph)

Sections are ordered by Gibbs step.

### Functions used for checks:
1. **Trace plot**: it plots a scalar quantity over post-burnin iterations. It is used to check *stationarity*: the series should fluctuate around a stable mean with no visible trend. 

2. **ACF grid**: it plots the autocorrelation function for each parameter whose corresponding entry in `allowed_mask` is 1. 

3. **ESS heatmap**: computes and plots the Effective Sample Size for each parameter.

4. **Posterior heatmap**: plots the posterior mean of each entry across iterations.
    - For binary parameters (graphs): the posterior mean is defined as the inclusion probability shown on a 0-1 scale with a red colormap.
    - For continuous parameters (Sigma_u, Phi): the posterior mean is defined as the average coefficient value, shown on a diverging colormap centered at zero.

### 1. Trace Plot
**Arguments**:
- `scalar_series` : (N_KEEP,) array with one number per iteration (e.g. number of active edges, log-determinant of Sigma_u)
- `title`         : string shown as plot title

In [ ]:
def plot_trace(scalar_series: np.ndarray, 
               title: str):
    
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(scalar_series, lw=0.7, color="steelblue")
    ax.axhline(scalar_series.mean(), color="red", lw=1,
               linestyle="--", label=f"mean = {scalar_series.mean():.2f}")
    ax.set_xlabel("Post-burnin iteration")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

### 2. ACF grid
**Arguments**:
- `samples_3d` : (ny, ny, N_KEEP) array with full chain for all parameters
- `allowed_mask` : (ny, ny) binary array with which entries to compute
- `title` : string shown as plot title

In [ ]:
def plot_acf_grid(samples_3d: np.ndarray, 
                  allowed_mask: np.ndarray, 
                  title: str):
    
    N_KEEP     = samples_3d.shape[2]
    entries    = list(zip(*np.where(allowed_mask == 1)))
    n_entries  = len(entries)
    n_cols     = 4
    n_rows     = (n_entries + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 3.5, n_rows * 2.5),
                             sharey=True)
    axes = axes.flatten()

    for idx, (i, j) in enumerate(entries):
        chain    = samples_3d[i, j, :].astype(float)
        acf_vals = acf(chain, nlags=40, fft=True)
        axes[idx].bar(range(len(acf_vals)), acf_vals, width=0.8, color="steelblue")
        axes[idx].axhline(0, color="black", lw=0.5)
        axes[idx].axhline( 1.96 / np.sqrt(N_KEEP), color="red", lw=0.8, linestyle="--")
        axes[idx].axhline(-1.96 / np.sqrt(N_KEEP), color="red", lw=0.8, linestyle="--")
        axes[idx].set_title(f"{j} → {i}", fontsize=9)
        axes[idx].set_ylim(-0.3, 1.05)

    for idx in range(n_entries, len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()

### 3. ESS heatmap
**ESS formula**: 
$$
\frac{N}{1 + 2 \cdot\sum (\text{positive autocorrelations})}
$$
it uses Greyer's initial monotone criterion: it stops summing autocorrelarions at the first negative value to avoid noise inflation

**Arguments**:
- `samples_3d` : (ny, ny, N_KEEP) array with full chain for all parameters
- `allowed_mask` : (ny, ny) binary array with which entries to compute
- `title` : string shown as plot title

In [ ]:
def plot_ess_heatmap(samples_3d: np.ndarray, 
                     allowed_mask: np.ndarray, 
                     title: str):
    
    def _ess(chain: np.ndarray, max_lags: int = 100) -> float:
        n        = len(chain)
        acf_vals = acf(chain.astype(float), nlags=max_lags, fft=True)
        rho_sum  = 0.0
        for k in range(1, len(acf_vals)):
            if acf_vals[k] < 0:
                break
            rho_sum += acf_vals[k]
        return n / (1 + 2 * rho_sum)

    ny         = samples_3d.shape[0]
    N_KEEP     = samples_3d.shape[2]
    entries    = list(zip(*np.where(allowed_mask == 1)))
    ess_matrix = np.zeros((ny, ny))

    for i, j in entries:
        ess_matrix[i, j] = _ess(samples_3d[i, j, :])

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(ess_matrix, cmap="RdYlGn", aspect="auto", vmin=0, vmax=N_KEEP)
    plt.colorbar(im, ax=ax, label="ESS")
    ax.set_title(title)
    ax.set_xlabel("Parent j")
    ax.set_ylabel("Child i")

    for i in range(ny):
        for j in range(ny):
            if allowed_mask[i, j] == 1:
                ax.text(j, i, f"{int(ess_matrix[i,j])}", ha="center", va="center",
                        fontsize=7,
                        color="white" if ess_matrix[i,j] < N_KEEP * 0.3 else "black")
    plt.tight_layout()
    plt.show()

    ess_vals = [ess_matrix[i, j] for i, j in entries]
    print(f"ESS summary — min={min(ess_vals):.0f}  "
          f"mean={np.mean(ess_vals):.0f}  "
          f"max={max(ess_vals):.0f}  "
          f"arcs with ESS<100: {sum(v < 100 for v in ess_vals)}/{len(entries)}")

### 4. Posterior heathmap
**Arguments**:
- `samples_3d` : (ny, ny, N_KEEP) array with full chain for all parameters
- `allowed_mask` : (ny, ny) binary array with which entries to compute
- `title` : string shown as plot title
- `binary` : True for graphs (0/1 chain, inclusion prob colormap), False for continuous parameters (diverging colormap)

In [ ]:
def plot_posterior_heatmap(samples_3d: np.ndarray, 
                           title: str,
                           allowed_mask: np.ndarray = None,
                           binary: bool = False):
   
    post_mean = samples_3d.mean(axis=2)

    if allowed_mask is not None:
        post_mean = np.where(allowed_mask == 1, post_mean, np.nan)

    cmap   = "YlOrRd" if binary else "RdBu_r"
    vmin   = 0        if binary else None
    vmax   = 1        if binary else None

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(post_mean, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax,
                 label="Inclusion probability" if binary else "Posterior mean")
    ax.set_title(title)
    ax.set_xlabel("Parent j")
    ax.set_ylabel("Child i")

    ny = post_mean.shape[0]
    for i in range(ny):
        for j in range(ny):
            v = post_mean[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        fontsize=7,
                        color="white" if (binary and v > 0.6) else "black")
    plt.tight_layout()
    plt.show()

## Overall chain diagnostics

## Step 1 diagnostics - G0

In [ ]:
G0_samples  = np.load(OUTPUT_DIR / "G0_samples.npy")
G0_expanded = np.load(OUTPUT_DIR / "G0_expanded.npy") 

ny, _, N_KEEP = G0_samples.shape

In [ ]:
plot_trace(G0_samples.sum(axis=(0,1)).astype(float), 
           "G0 — active edges per iteration")

In [ ]:
plot_acf_grid(G0_samples, 
              G0_expanded, 
              "G0 — ACF per arc")

In [ ]:
plot_ess_heatmap(G0_samples, 
                 G0_expanded, 
                 "G0 — ESS per arc")

In [ ]:
plot_posterior_heatmap(G0_samples, 
                       "G0 — posterior inclusion probabilities",
                       allowed_mask=G0_expanded, 
                       binary=True)

## Step 2 diagnostics

## Step 3 diagnostics

## Step 4 diagnostics

## Step 5 diagnostics

## Step 6 diagnostics